# VAE: 分布として潜在空間を学ぶ

VAEは、入力を潜在変数の分布へ写し、その分布からサンプルしたzをDecoderで復元する生成モデルである。


## このノートの読み方

想定読者: Autoencoder、正規分布、MSE、KL divergenceの入口を理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

Autoencoderは1点の潜在表現を使う。VAEは`mu`と`logvar`で分布を出し、生成に使いやすい潜在空間を学ぶ。


## 到達目標

- Encoderが分布を出す意味を説明できる
- reparameterization trickを説明できる
- ELBOの再構成項とKL項を区別できる


## 重要語句

- `ELBO`: 対数尤度の下界
- `KL`: 推論分布と事前分布のずれ
- `reparameterization`: サンプリングを微分可能な形へ書き換える工夫


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| x | (B, x_dim) | 入力 |
| mu/logvar | (B, z_dim) | 潜在分布 |
| reconstruction | (B, x_dim) | 復元 |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| Autoencoderとの差分 | 通常のAutoencoderは潜在点を出す。VAEは`mu`と`logvar`で分布を出し、そこからサンプルする。 |
| 負のELBO | 数式ではELBOを最大化する。実装では`recon_loss + KL`という負のELBO相当を最小化する。 |
| likelihood仮定 | MSE再構成は、Decoderの出力を平均、分散固定のガウス尤度とみなす簡略化である。 |
| 生成 | 学習後は`z ~ N(0,I)`をDecoderへ入れる。観測できない細胞状態や測定条件を潜在空間として歩く見方ができる。 |


## Latent variable model

観測xの背後に潜在zがあると考える。

$$
p_\theta(x,z)=p(z)p_\theta(x|z)
$$


## Reparameterization

ランダム性をepsilonへ分離し、muとlogvarへ勾配を通す。

$$
z=\mu+\exp(0.5\log\sigma^2)\odot\epsilon,\quad \epsilon\sim\mathcal{N}(0,I)
$$


## ELBO

復元と潜在空間の整え方の綱引きである。

$$
\mathcal{L}=\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]-\mathrm{KL}(q_\phi(z|x)\|p(z))
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
mu = torch.tensor([[0.2, -0.1], [0.0, 0.5]])
logvar = torch.tensor([[0.1, -0.3], [0.2, 0.0]])
eps = torch.randn_like(mu)
z = mu + torch.exp(0.5 * logvar) * eps
kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
print("z:", z.round(decimals=3))
print("KL:", kl.round(decimals=3))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/vae_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/vae_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/vae_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="VAE: 分布として潜在空間を学ぶ difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### encoder distribution animation

- 学習目標: Encoderが点ではなく分布を出す
- 誤解の防止: 潜在表現が1点だと思う

対応する式:

$$
q_\phi(z|x)=\mathcal{N}(\mu,\mathrm{diag}(\sigma^2))
$$


<p><a href="../demos/vae_encoder_distribution.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/vae_encoder_distribution.html</code>）</p>
<iframe
  src="../demos/vae_encoder_distribution.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="encoder distribution animation"
></iframe>


### reparameterization animation

- 学習目標: epsilonからzを作る
- 誤解の防止: サンプリングで勾配が止まると思う

対応する式:

$$
z=\mu+\sigma\odot\epsilon
$$


<p><a href="../demos/vae_reparameterization.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/vae_reparameterization.html</code>）</p>
<iframe
  src="../demos/vae_reparameterization.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="reparameterization animation"
></iframe>


### ELBO balance animation

- 学習目標: 再構成項とKL項の綱引き
- 誤解の防止: 復元だけ頑張ればよいと思う

対応する式:

$$
loss=recon+KL
$$


<p><a href="../demos/vae_elbo_balance.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/vae_elbo_balance.html</code>）</p>
<iframe
  src="../demos/vae_elbo_balance.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="ELBO balance animation"
></iframe>


### latent interpolation animation

- 学習目標: 潜在空間を歩くと出力が変わる
- 誤解の防止: zが意味を持たないと思う

対応する式:

$$
z(\lambda)=(1-\lambda)z_a+\lambda z_b
$$


<p><a href="../demos/vae_latent_interpolation.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/vae_latent_interpolation.html</code>）</p>
<iframe
  src="../demos/vae_latent_interpolation.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="latent interpolation animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`VAE: 分布として潜在空間を学ぶ`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyVAEDataset(Dataset):
    def __init__(self, n_samples: int = 32, x_dim: int = 4) -> None:
        self.features = torch.randn(n_samples, x_dim)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"features": self.features[index]}


class TrainerVAE(nn.Module):
    def __init__(self, x_dim: int = 4, z_dim: int = 2) -> None:
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(x_dim, 16), nn.ReLU())
        self.mu = nn.Linear(16, z_dim)
        self.logvar = nn.Linear(16, z_dim)
        self.decoder = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, x_dim))

    def forward(self, features: torch.Tensor) -> dict[str, torch.Tensor]:
        h = self.encoder(features)
        mu = self.mu(h)
        logvar = self.logvar(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        reconstruction = self.decoder(z)
        recon_loss = torch.mean((reconstruction - features) ** 2)
        kl_per_sample = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
        kl_loss = torch.mean(kl_per_sample)
        loss = recon_loss + kl_loss
        return {"loss": loss, "logits": reconstruction}


training_args = TrainingArguments(
    output_dir="./results/vae_trainer_demo",
    max_steps=1,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

trainer = Trainer(model=TrainerVAE(), args=training_args, train_dataset=TinyVAEDataset())
train_output = trainer.train()
print("VAE Trainer loss:", train_output.training_loss)


with torch.no_grad():
    z = torch.randn(3, 2)
    generated = trainer.model.decoder(z)
print("prior sample shape:", generated.shape)


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- priorからsampleする
- beta-VAEを試す
- IWAE章でKサンプルへ進む


## 確認問題

- VAEのEncoderは何を出力するか。
- KL項をなくすと何が起きるか。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
